In [1]:
import requests
from bs4 import BeautifulSoup
import csv
import time
from urllib.parse import urljoin, urlparse

BASE_URL = "https://www.bangalorestartupmap.com"
STARTUPS_LIST_URL = "https://www.bangalorestartupmap.com/startups"
OUTPUT_CSV = "bangalore_startups_details.csv"

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0 Safari/537.36"
    )
}

def get_soup(url):
    resp = requests.get(url, headers=HEADERS, timeout=10)
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")

def extract_companies_from_list():
    soup = get_soup(STARTUPS_LIST_URL)
    companies = []
    
    # All company links are typically in <li><a href="/company/...">Name</a></li>
    for a in soup.select("li a"):
        href = a.get("href")
        if not href:
            continue
        path = urlparse(href).path
        if not path.startswith("/company/"):
            continue
        name = a.get_text(strip=True)
        if not name:
            continue
        companies.append((name, urljoin(BASE_URL, href)))
    
    return companies

def extract_text_after_label(soup, label):
    """
    Try to find text after a label like 'Sector', 'Founded', 'Team size', etc.
    Works with patterns like:
      - <li>- **Sector** Edtech</li>
      - <li><b>Sector</b>: Edtech</li>
      - <p>Sector: Edtech</p>
    Returns cleaned string or empty string if not found.
    """
    label_lower = label.lower()
    
    # Try in <li> tags first
    for li in soup.select("li"):
        text = li.get_text(" ", strip=True)
        if label_lower in text.lower():
            # Split on label and take what's after
            parts = text.lower().split(label_lower, 1)
            if len(parts) < 2:
                continue
            after = parts[1].strip()
            # Remove leading punctuation like ':', '–', etc.
            after = after.lstrip(":–- ").strip()
            # Sometimes there's extra text after; take first meaningful chunk
            # If it looks like a sentence, we still keep it.
            if after:
                return after
    
    # Fallback: search in <p> and <div>
    for tag in soup.find_all(["p", "div"]):
        text = tag.get_text(" ", strip=True)
        if label_lower in text.lower():
            parts = text.lower().split(label_lower, 1)
            if len(parts) < 2:
                continue
            after = parts[1].strip().lstrip(":–- ").strip()
            if after:
                return after
    
    return ""

def extract_company_detail(company_url):
    soup = get_soup(company_url)
    
    # Company name: usually in <h1>
    h1 = soup.find("h1")
    name = h1.get_text(strip=True) if h1 else ""
    
    # Extract fields
    sector = extract_text_after_label(soup, "Sector")
    founded = extract_text_after_label(soup, "Founded")
    team_size = extract_text_after_label(soup, "Team size")
    
    # Normalize some common variations
    if not sector:
        # Sometimes it's "Industry" instead of "Sector"
        sector = extract_text_after_label(soup, "Industry")
    
    return {
        "name": name,
        "sector": sector,
        "founded": founded,
        "team_size": team_size,
    }

def main():
    print("Fetching companies list...")
    companies = extract_companies_from_list()
    print(f"Found {len(companies)} companies.")
    
    results = []
    
    for i, (name_preview, url) in enumerate(companies, start=1):
        print(f"[{i}/{len(companies)}] {name_preview}")
        try:
            data = extract_company_detail(url)
            
            # Fallback name if <h1> missing
            if not data["name"]:
                data["name"] = name_preview
            
            results.append({
                "name": data["name"],
                "sector": data["sector"],
                "founded": data["founded"],
                "team_size": data["team_size"],
                "url": url,
            })
        except Exception as e:
            print(f"  Error: {e}")
            results.append({
                "name": name_preview,
                "sector": "",
                "founded": "",
                "team_size": "",
                "url": url,
            })
        
        # Small delay to avoid hammering the server
        time.sleep(0.3)
    
    # Save to CSV
    with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
        fieldnames = ["name", "sector", "founded", "team_size", "url"]
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(results)
    
    print(f"\nDone! Saved to {OUTPUT_CSV}")

if __name__ == "__main__":
    main()

Fetching companies list...
Found 1085 companies.
[1/1085] AARUM
[2/1085] ABB India
[3/1085] Accel India
[4/1085] Acceldata
[5/1085] Accenture India
[6/1085] Actyv.ai
[7/1085] Acviss Technologies
[8/1085] Addverb Technologies
[9/1085] Adobe Systems India
[10/1085] Adopt AI
[11/1085] AdSocial.ai
[12/1085] AEOS
[13/1085] Aerchain
[14/1085] Aereo
[15/1085] Aetos Innovation Labs
[16/1085] Aforro
[17/1085] Agilent Technologies India
[18/1085] AgWise
[19/1085] Aindra Systems
[20/1085] Airbound
[21/1085] Airbus India
[22/1085] Airmeet
[23/1085] Airtribe
[24/1085] Aisle
[25/1085] Akamai Technologies India
[26/1085] Akshayakalpa Organic
[27/1085] Algen AI
[28/1085] Allegion India
[29/1085] Allo Health
[30/1085] AlmaBetter
[31/1085] Alpha Wave Global
[32/1085] Alpify Technologies
[33/1085] Alt Carbon
[34/1085] ALTEN Calsoft Labs
[35/1085] Altigreen
[36/1085] AltM
[37/1085] Amagi
[38/1085] Amazon Development Centre India
[39/1085] Ambitio
[40/1085] AMD India
[41/1085] Analog Devices India
[42/1085